In [ ]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import PolynomialFeatures
import copy
from tqdm.notebook import tqdm

from sklearn.metrics import average_precision_score, average_precision_score
from sklearn.preprocessing import label_binarize
import numpy as np
import random
from itertools import combinations
from collections import defaultdict

# Set random seed for reproducibility
random.seed(42)
np.random.seed(42)

# set to not print pandas warnings
pd.options.mode.chained_assignment = None  # default='warn'

In [10]:
# Load data

df_train = pd.read_csv("../../data_new/splits/train_extra_cols.csv")
# df_train = df_train[df_train["mask"] == "y"]
df_train = df_train[~df_train["bg_derivative"].isna()]
df_train = df_train[~df_train["cso_derivative"].isna()]

df_val = pd.read_csv("../../data_new/splits/val_extra_cols.csv")
# df_val = df_val[df_val["mask"] == "y"]
df_val = df_val[~df_val["bg_derivative"].isna()]
df_val = df_val[~df_val["cso_derivative"].isna()]

df_test = pd.read_csv("../../data_new/metadata_extra_cols.csv")
# df_test = df_test[df_test["mask"] == "y"]
df_test = df_test[~df_test["bg_derivative"].isna()]
df_test = df_test[~df_test["cso_derivative"].isna()]

# Prepare data

def min_max_normalize(df, columns):
    min_vals = df[columns].min()
    max_vals = df[columns].max()
    normalized_df = df.copy()
    normalized_df[columns] = (df[columns] - min_vals) / (max_vals - min_vals)
    return normalized_df, min_vals, max_vals

def apply_min_max(df, columns, min_vals, max_vals):
    normalized_df = df.copy()
    normalized_df[columns] = (df[columns] - min_vals) / (max_vals - min_vals)
    return normalized_df

# Code sex: M=0, F=1

df_train["sex"][df_train["sex"]=="M"] = 0
df_train["sex"][df_train["sex"]=="F"] = 1
df_val["sex"][df_val["sex"]=="M"] = 0
df_val["sex"][df_val["sex"]=="F"] = 1
df_test["sex"][df_test["sex"]=="M"] = 0
df_test["sex"][df_test["sex"]=="F"] = 1

# Normalize the data

columns_to_normalize = [
    "age",
    "vols_gt",
    "counts_gt",
    "BG_vols_gt",
    "BG_counts_gt",
    "CSO_vols_gt",
    "CSO_counts_gt",
    "BG_max_vols_gt",
    "BG_max_counts_gt",
    "CSO_max_vols_gt",
    "CSO_max_counts_gt",
    "vols_pred",
    "counts_pred",
    "BG_vols_pred",
    "BG_counts_pred",
    "CSO_vols_pred",
    "CSO_counts_pred",
    "BG_max_vols_pred",
    "BG_max_counts_pred",
    "CSO_max_vols_pred",
    "CSO_max_counts_pred"
]

df_train, min_vals, max_vals = min_max_normalize(df_train, columns_to_normalize)
df_val = apply_min_max(df_val, columns_to_normalize, min_vals, max_vals)
df_test = apply_min_max(df_test, columns_to_normalize, min_vals, max_vals)

# iterator func

def all_combinations(lst):
    for r in range(1, len(lst)+1):
        for combo in combinations(lst, r):
            yield list(combo)

### Logistic regression - polynomial

In [13]:
def polynomial_transform(X, degree, include_bias=False):
    poly = PolynomialFeatures(degree=degree, include_bias=include_bias)
    return poly.fit_transform(X)

In [14]:
max_AP_BG = 0
best_features_BG = []
best_model_BG = None
best_deg_BG = 1

all_features = ["sex"] + [x for x in columns_to_normalize if "CSO" not in x]
all_features = [x for x in all_features if "_gt" not in x]

for features in tqdm(all_combinations(all_features), total=2**len(all_features)-1):
    
    for deg in [1,2,3]:
    
        LR_BG = LogisticRegression(class_weight="balanced", max_iter=5000, random_state=42)
        X_train = polynomial_transform(df_train[features].values, deg, False)
        LR_BG.fit(X=X_train, y=df_train[["bg_derivative"]].values.squeeze())
        
        y_true=df_val[["bg_derivative"]].values.squeeze()
        X_val = polynomial_transform(df_val[features].values, deg, False)
        y_pred=LR_BG.predict(X=X_val)
        
        AP = average_precision_score(label_binarize(y_true, classes=[1,2,3]), label_binarize(y_pred, classes=[1,2,3]), average="macro")
        
        if AP > max_AP_BG:
            max_AP_BG = AP
            best_features_BG = copy.deepcopy(features)
            best_model_BG = copy.deepcopy(LR_BG)
            best_deg_BG = deg
        
print(f"Best AP: {max_AP_BG}     features: {best_features_BG},        degree: {best_deg_BG}")

  0%|          | 0/255 [00:00<?, ?it/s]

Best AP: 0.576046443775729     features: ['age', 'BG_vols_pred', 'BG_max_counts_pred'],        degree: 2


In [15]:
max_AP_CSO = 0
best_features_CSO = []
best_model_CSO = None
best_deg_CSO = 1

all_features = ["sex"] + [x for x in columns_to_normalize if "BG" not in x]
all_features = [x for x in all_features if "_gt" not in x]

for features in tqdm(all_combinations(all_features), total=2**len(all_features)-1):
    
    for deg in [1,2,3]:
    
        LR_CSO = LogisticRegression(class_weight="balanced", max_iter=5000, random_state=42)
        X_train = polynomial_transform(df_train[features].values, deg, False)
        LR_CSO.fit(X=X_train, y=df_train[["cso_derivative"]].values.squeeze())
        
        y_true=df_val[["cso_derivative"]].values.squeeze()
        X_val = polynomial_transform(df_val[features].values, deg, False)
        y_pred=LR_CSO.predict(X=X_val)
        
        AP = average_precision_score(label_binarize(y_true, classes=[1,2,3]), label_binarize(y_pred, classes=[1,2,3]), average="macro")
        
        if AP > max_AP_CSO:
            max_AP_CSO = AP
            best_features_CSO = copy.deepcopy(features)
            best_model_CSO = copy.deepcopy(LR_CSO)
            best_deg_CSO = deg
        
print(f"Best AP: {max_AP_CSO}     features: {best_features_CSO},        degree: {best_deg_CSO}")

  0%|          | 0/255 [00:00<?, ?it/s]

Best AP: 0.4870152789672069     features: ['sex', 'counts_pred', 'CSO_max_vols_pred'],        degree: 3


In [16]:
output = defaultdict(list)

for row in df_test.iterrows():
    output["id"].append(row[1]["id"])
    X_test_BG = polynomial_transform(row[1][best_features_BG].values[None, :], best_deg_BG, False)
    X_test_CSO = polynomial_transform(row[1][best_features_CSO].values[None, :], best_deg_CSO, False)
    pred_BG = best_model_BG.predict_proba(X=X_test_BG)
    pred_CSO = best_model_CSO.predict_proba(X=X_test_CSO)
    for i in range(3):
        output[f"bg_derivative_{i+1}"].append(pred_BG[0,i])
    for i in range(3):
        output[f"cso_derivative_{i+1}"].append(pred_CSO[0,i])
        
df = pd.DataFrame(output)
df.to_csv("../../results_new/LR_all/predictions.csv", index=False)